# Random Forest

In [1]:
import numpy as np
import pandas as pd 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score, average_precision_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import joblib

## Data Loading - No Covid

In [2]:
X_train = pd.read_parquet('artifacts/X_train_no_covid.parquet')
X_val = pd.read_parquet('artifacts/X_val_no_covid.parquet')
X_test = pd.read_parquet('artifacts/X_test_no_covid.parquet')

y_train = pd.read_parquet('artifacts/y_train.parquet')
y_val = pd.read_parquet('artifacts/y_val.parquet')
y_test = pd.read_parquet('artifacts/y_test.parquet')

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(2186803, 24) (349591, 24) (698587, 24)
(2186803, 1) (349591, 1) (698587, 1)


### Base RF model

In [3]:
rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_base.fit(X_train, y_train)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [4]:
# find the best threshold using validation dataset
y_proba_val = rf_base.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.01, 0.5, 50)
best_f2 = 0
best_threshold_base = 0

for t in thresholds:
    y_pred_val = (y_proba_val >= t).astype(int)
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    if f2 > best_f2:
        best_f2 = f2
        best_threshold_base = t

print("\nBest threshold (val):", best_threshold_base)
print("Best F2 (val):", best_f2)
print("PR-AUC (val):", average_precision_score(y_val, y_proba_val))


Best threshold (val): 0.04
Best F2 (val): 0.1941194911478872
PR-AUC (val): 0.09830793877178727


In [6]:
# get the final evaluation metrics from the testing dataset
y_proba_test_base = rf_base.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test_base >= best_threshold_base).astype(int)

pr_auc_base = average_precision_score(y_test, y_proba_test_base)
f2_base = fbeta_score(y_test, y_pred_test, beta=2)

print("PR-AUC of best model: ", pr_auc_base)
print("F2 of best model: ", f2_base)

PR-AUC of best model:  0.04144920774370401
F2 of best model:  0.1330580408350539


In [7]:
accuracy_base  = accuracy_score(y_test, y_pred_test)
precision_base = precision_score(y_test, y_pred_test)
recall_base    = recall_score(y_test, y_pred_test)
f1_base        = f1_score(y_test, y_pred_test)
f2_base        = fbeta_score(y_test, y_pred_test, beta=2)

roc_auc_base = roc_auc_score(y_test, y_proba_test_base)
pr_auc_base  = average_precision_score(y_test, y_proba_test_base)

print("Accuracy of base model: ", accuracy_base)
print("Precision of base model:", precision_base)
print("Recall of base model:   ", recall_base)
print("F1 of best model:       ", f1_base)
print("F2 of best model:       ", f2_base)
print("ROC-AUC of best model:  ", roc_auc_base)
print("PR-AUC of best model:   ", pr_auc_base)

Accuracy of base model:  0.8078965111002638
Precision of base model: 0.03844108603004077
Recall of base model:    0.34590846047156726
F1 of best model:        0.06919272838247432
F2 of best model:        0.1330580408350539
ROC-AUC of best model:   0.6251256658919689
PR-AUC of best model:    0.04144920774370401


### Create sample dataset for tuning

This is needed for the hyperparameter tuning which will try 20 combinations. And it safe to do this because the params used usually stay similar even if the dataset size changes. <br>
Base model is not included because the final metric will still be done on the full dataset. 

In [8]:
sample_size = 200000

X_sample = X_train.sample(sample_size, random_state=42)
y_sample = y_train.loc[X_sample.index]

### Hyperparameter tuning

In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [10]:
# parameters that can be tuned
param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [20, 30, None],
    "max_features": ["sqrt", "log2", 0.5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "class_weight": ['balanced']
}

In [11]:
# try 70 random combinations
search = RandomizedSearchCV(
    rf_base,
    param_dist,
    n_iter=70,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=42
)

search.fit(X_sample, y_sample)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_iter=70, n_jobs=-1,
                   param_distributions={'class_weight': ['balanced'],
                                        'max_depth': [20, 30, None],
                                        'max_features': ['sqrt', 'log2', 0.5],
                                        'min_samples_leaf': [1, 2, 5],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [200, 400, 600]},
                   random_state=42, scoring='average_precision')

In [12]:
#search the best and retrieve matrics
print("Best Parameters: ", search.best_params_)
print("Best PR-AUC: ", search.best_score_)

Best Parameters:  {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}
Best PR-AUC:  0.22164072337392593


### Train Best Model on Full Training dataset

In [13]:
best_rf = RandomForestClassifier(
    **search.best_params_,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(class_weight='balanced', min_samples_leaf=2,
                       min_samples_split=5, n_estimators=400, n_jobs=-1,
                       random_state=42)

In [14]:
y_proba_val = best_rf.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.01, 0.5, 50)
best_f2 = 0
best_threshold_best = 0

for t in thresholds:
    y_pred_val = (y_proba_val >= t).astype(int)
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    if f2 > best_f2:
        best_f2 = f2
        best_threshold = t

print("\nBest threshold (val):", best_threshold_best)
print("Best F2 (val):", best_f2)
print("PR-AUC (val):", average_precision_score(y_val, y_proba_val))


Best threshold (val): 0
Best F2 (val): 0.22799175392402996
PR-AUC (val): 0.12843773024663752


In [16]:
y_proba_test_best = best_rf.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test_best >= best_threshold).astype(int)

pr_auc_best = average_precision_score(y_test, y_proba_test_best)
f2_best = fbeta_score(y_test, y_pred_test, beta=2)

print("PR-AUC of best model: ", pr_auc_best)
print("F2 of best model: ", f2_best)

PR-AUC of best model:  0.047278315399000154
F2 of best model:  0.14019990537023894


In [17]:
results = pd.DataFrame({
    "Model": ["Baseline RF", "Tuned RF"], 
    "Threshold": [best_threshold_base, best_threshold_best],
    "PR-AUC": [pr_auc_base, pr_auc_best],
    "F2": [f2_base, f2_best]
})

print(results)

         Model  Threshold    PR-AUC        F2
0  Baseline RF       0.04  0.041449  0.133058
1     Tuned RF       0.00  0.047278  0.140200


In [18]:
joblib.dump(best_rf, "rf_balanced_final.pkl")

['rf_balanced_final.pkl']

-------
## Data Loading - Covid

In [19]:
X_train = pd.read_parquet('artifacts/X_train_with_covid.parquet')
X_val = pd.read_parquet('artifacts/X_val_with_covid.parquet')
X_test = pd.read_parquet('artifacts/X_test_with_covid.parquet')

y_train = pd.read_parquet('artifacts/y_train.parquet')
y_val = pd.read_parquet('artifacts/y_val.parquet')
y_test = pd.read_parquet('artifacts/y_test.parquet')

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(2186803, 25) (349591, 25) (698587, 25)
(2186803, 1) (349591, 1) (698587, 1)


### Base RF model

In [22]:
rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_base.fit(X_train, y_train)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [23]:
# find the best threshold using validation dataset
y_proba_val = rf_base.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.01, 0.5, 50)
best_f2 = 0
best_threshold_base = 0

for t in thresholds:
    y_pred_val = (y_proba_val >= t).astype(int)
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    if f2 > best_f2:
        best_f2 = f2
        best_threshold_base = t

print("\nBest threshold (val):", best_threshold_base)
print("Best F2 (val):", best_f2)
print("PR-AUC (val):", average_precision_score(y_val, y_proba_val))


Best threshold (val): 0.05
Best F2 (val): 0.19623028832695716
PR-AUC (val): 0.09921497588665515


In [24]:
# get the final evaluation metrics from the testing dataset
y_proba_test_base = rf_base.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test_base >= best_threshold).astype(int)

pr_auc_base = average_precision_score(y_test, y_proba_test_base)
f2_base = fbeta_score(y_test, y_pred_test, beta=2)

print("PR-AUC of best model: ", pr_auc_base)
print("F2 of best model: ", f2_base)

PR-AUC of best model:  0.06007169485823828
F2 of best model:  0.1488931887602984


In [25]:
accuracy_base  = accuracy_score(y_test, y_pred_test)
precision_base = precision_score(y_test, y_pred_test)
recall_base    = recall_score(y_test, y_pred_test)
f1_base        = f1_score(y_test, y_pred_test)
f2_base        = fbeta_score(y_test, y_pred_test, beta=2)

roc_auc_base = roc_auc_score(y_test, y_proba_test_base)
pr_auc_base  = average_precision_score(y_test, y_proba_test_base)

print("Accuracy of base model: ", accuracy_base)
print("Precision of base model:", precision_base)
print("Recall of base model:   ", recall_base)
print("F1 of best model:       ", f1_base)
print("F2 of best model:       ", f2_base)
print("ROC-AUC of best model:  ", roc_auc_base)
print("PR-AUC of best model:   ", pr_auc_base)

Accuracy of base model:  0.9513031304619182
Precision of base model: 0.09990609561915649
Recall of base model:    0.16969486823855756
F1 of best model:        0.12576773828797574
F2 of best model:        0.1488931887602984
ROC-AUC of best model:   0.6592802297417847
PR-AUC of best model:    0.06007169485823828


### Hyperparameter Tuning

In [26]:
sample_size = 200000

X_sample_covid = X_train.sample(sample_size, random_state=42)
y_sample_covid = y_train.loc[X_sample.index]

In [27]:
# parameters that can be tuned
param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [20, 30, None],
    "max_features": ["sqrt", "log2", 0.5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "class_weight": ['balanced']
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [28]:
# try 70 random combinations
search = RandomizedSearchCV(
    rf_base,
    param_dist,
    n_iter=70,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=42
)

search.fit(X_sample_covid, y_sample_covid)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_iter=70, n_jobs=-1,
                   param_distributions={'class_weight': ['balanced'],
                                        'max_depth': [20, 30, None],
                                        'max_features': ['sqrt', 'log2', 0.5],
                                        'min_samples_leaf': [1, 2, 5],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [200, 400, 600]},
                   random_state=42, scoring='average_precision')

In [29]:
#search the best and retrieve matrics
print("Best Parameters: ", search.best_params_)
print("Best PR-AUC: ", search.best_score_)

Best Parameters:  {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}
Best PR-AUC:  0.2573495500230422


### Train Best Model on Full Training dataset

In [30]:
best_rf_covid = RandomForestClassifier(
    **search.best_params_,
    random_state=42,
    n_jobs=-1
)

best_rf_covid.fit(X_train, y_train)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(class_weight='balanced', min_samples_leaf=2,
                       min_samples_split=5, n_estimators=400, n_jobs=-1,
                       random_state=42)

In [31]:
y_proba_val = best_rf_covid.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.01, 0.5, 50)
best_f2 = 0
best_threshold_best = 0

for t in thresholds:
    y_pred_val = (y_proba_val >= t).astype(int)
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    if f2 > best_f2:
        best_f2 = f2
        best_threshold_best = t

print("\nBest threshold (val):", best_threshold_best)
print("Best F2 (val):", best_f2)
print("PR-AUC (val):", average_precision_score(y_val, y_proba_val))


Best threshold (val): 0.060000000000000005
Best F2 (val): 0.22785622593068036
PR-AUC (val): 0.12498761098675618


In [32]:
y_proba_test_best = best_rf_covid.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test_best >= best_threshold).astype(int)

pr_auc_best = average_precision_score(y_test, y_proba_test_best)
f2_best = fbeta_score(y_test, y_pred_test, beta=2)

print("PR-AUC of best model: ", pr_auc_best)
print("F2 of best model: ", f2_best)

PR-AUC of best model:  0.07424043603666783
F2 of best model:  0.19240396489003103


In [33]:
accuracy_best  = accuracy_score(y_test, y_pred_test)
precision_best = precision_score(y_test, y_pred_test)
recall_best    = recall_score(y_test, y_pred_test)
f1_best        = f1_score(y_test, y_pred_test)
f2_best        = fbeta_score(y_test, y_pred_test, beta=2)

roc_auc_best = roc_auc_score(y_test, y_proba_test_best)
pr_auc_best  = average_precision_score(y_test, y_proba_test_best)

print("Accuracy of best model: ", accuracy_best)
print("Precision of best model:", precision_best)
print("Recall of best model:   ", recall_best)
print("F1 of best model:       ", f1_best)
print("F2 of best model:       ", f2_best)
print("ROC-AUC of best model:  ", roc_auc_best)
print("PR-AUC of best model:   ", pr_auc_best)

Accuracy of best model:  0.9035653397500956
Precision of best model: 0.07418130348066654
Recall of best model:    0.31983356449375866
F1 of best model:        0.12043033214957176
F2 of best model:        0.19240396489003103
ROC-AUC of best model:   0.6948705174153214
PR-AUC of best model:    0.07424043603666783


In [34]:
joblib.dump(best_rf_covid, "rf_final_covid.pkl")

['rf_final_covid.pkl']

In [35]:
results = pd.DataFrame({
    "Model": ["Baseline RF", "Tuned RF"], 
    "Threshold": [best_threshold_base, best_threshold_best],
    "PR-AUC": [pr_auc_base, pr_auc_best],
    "F2": [f2_base, f2_best]
})

print(results)

         Model  Threshold    PR-AUC        F2
0  Baseline RF       0.05  0.060072  0.148893
1     Tuned RF       0.06  0.074240  0.192404
